In [ ]:
print("langgraph-tutorial-workbook-solutions.ipynb")


# LangGraph Tutorial: From Motivation to LLM-Powered Agents

This notebook introduces LangGraph, explains the problems it solves in LLM-powered applications, and walks through a runnable example using OpenAI, simple agents, and conditional routing.



## 1. Introduction

LangGraph is a framework for building **stateful, controllable, and reliable** LLM-powered applications using **graphs** rather than linear chains or free-form agent loops.

Instead of letting an LLM implicitly decide what happens next, LangGraph makes:
- Control flow **explicit**
- State **shared and typed**
- Execution **predictable and debuggable**

This makes it suitable for real-world, production-grade AI systems.


---

## 2. Problems with Earlier LLM-Powered Applications

Before LangGraph, most LLM apps were built with:
- Prompt-only calls
- Linear chains
- Autonomous agents

These approaches work for prototypes but struggle in production.

### 2.1 Implicit Control Flow
- Chains assume a fixed order
- Agents decide the next step internally
- Hard to enforce business rules or guardrails

### 2.2 Poor State Management
- State hidden in prompts
- Manual passing between steps
- Easy to overwrite or lose context

### 2.3 Uncontrolled Agent Loops
- Risk of infinite loops
- Difficult to debug
- Hard to reproduce failures

### 2.4 Low Observability
- No clear execution trace
- Hard to replay or inspect partial results

---

## 3. How LangGraph Solves These Problems

LangGraph introduces:
- Explicit graph-based orchestration
- Typed shared state
- Deterministic routing
- Controlled loops and retries

It applies classical software engineering principles—state machines, routing, orchestration—to LLM systems.

---

## 4. Modes of Using LangGraph

### 4.1 Single-Agent Graphs
Structured workflows replacing complex chains.

### 4.2 Conditional and Tool-Oriented Graphs
Decision-making systems with branching logic.

### 4.3 Multi-Agent Graphs
Role-based agents coordinated via a supervisor or orchestrator.

---

## 5. Typical and Industry-Standard Workflow

1. Define typed state
2. Implement small, testable nodes
3. Define explicit routing
4. Compile the graph
5. Execute with observability and checkpoints

This mirrors standard backend system design.

---

## 6. LLM-Powered Assistant with Agents and Conditional Routing

### Goal

Build a simple assistant that:
1. Classifies user intent using an LLM
2. Routes execution based on that intent
3. Uses a specialized agent to respond

---




### 6.1. Installation and Setup

Make sure you have your OpenAI API key registered in your account.


In [ ]:

!pip install -q langgraph langchain langchain-openai



### 6.2. Define the Shared State


In [1]:
from typing import TypedDict, Literal

In [2]:
class State(TypedDict):
    user_input: str
    intent: Literal["faq", "general", ""]
    response: str


### 6.3. Initialize the LLM


In [3]:
from langchain_openai import ChatOpenAI

In [4]:
f = open(r"E:\Lenovo Ideapad 330\company-material\digital-workforce-transformation\ai-upskill-5\key-vault\openai\api.key")
apikey = f.read()
f.close()


In [5]:

llm = ChatOpenAI(
    model="gpt-4o-mini",   # or your Groq model
    api_key=apikey,
    temperature=0,
)

##### Fallback

In [ ]:
path = r"E:\Lenovo Ideapad 330\company-material\ai-upskill\key-vault\huggingface\hugginface-test-token.txt"
f = open(path)
api_key = f.read()
f.close()

In [ ]:
from langchain_openai import ChatOpenAI

In [ ]:
    llm = ChatOpenAI(
        model="Qwen/Qwen2.5-72B-Instruct:fastest",
        openai_api_key=api_key,
        openai_api_base="https://router.huggingface.co/v1",
        temperature=0
    )


## 6.4. Classifier Agent


In [6]:

def classify_intent(state: State) -> State:
    prompt = f"""
    Classify the user request into one category:
    - faq
    - general

    User request:
    {state['user_input']}

    Respond with one word.
    """

    intent = llm.invoke(prompt).content.strip().lower()

    return {
        **state,
        "intent": intent
    }


## 6.5. FAQ Agent


In [7]:
def faq_agent(state: State) -> State:
    prompt = f"""
    You are a helpful FAQ assistant.
    Answer clearly and concisely:

    {state['user_input']}
    """

    response = llm.invoke(prompt).content

    return {
        **state,
        "response": response
    }



### 6.6 General Chat Agent


In [8]:
def general_agent(state: State) -> State:
    prompt = f"""
    You are a general conversational assistant.

    User message:
    {state['user_input']}
    """

    response = llm.invoke(prompt).content

    return {
        **state,
        "response": response
    }


### 6.7 Conditional Routing


In [9]:

def route_by_intent(state: State):
    if state["intent"] == "faq":
        return "faq"
    return "general"



### 6.8 Build the LangGraph


In [10]:
from langgraph.graph import StateGraph, END

graph = StateGraph(State)

graph.add_node("classifier", classify_intent)
graph.add_node("faq", faq_agent)
graph.add_node("general", general_agent)

graph.set_entry_point("classifier")

graph.add_conditional_edges(
    "classifier",
    route_by_intent,
    {
        "faq": "faq",
        "general": "general"
    }
)

graph.add_edge("faq", END)
graph.add_edge("general", END)

app = graph.compile()


## 6.9 Execute the Graph


In [13]:
result = app.invoke({
    "user_input": "who founded spaceX",
    "intent": "",
    "response": ""
})

print("Intent:", result["intent"])
print("Response:", result["response"])


Intent: faq
Response: SpaceX was founded by Elon Musk in March 2002.



### 6.10 Key Takeaways

- LLMs assist decision-making but do not control execution
- Explicit graphs improve reliability and debuggability
- LangGraph enables production-grade AI systems
